In [1]:
import pandas as pd
import numpy as np
import re
import string
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [2]:
train_df = pd.read_csv(
    'train_data.txt', 
    sep=':::', 
    engine='python', 
    names=['id', 'title', 'genre', 'description'], 
    index_col=0
)
test_df = pd.read_csv(
    'test_data.txt', 
    sep=':::', 
    engine='python', 
    names=['id', 'title', 'description'], 
    index_col=0
)
display(train_df.head())

,title,genre,description
id,,,
1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his do...
2,Cupid (1997),thriller,A brother and sister with a past incestuous r...
3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fie...
4,The Secret Sin (1915),drama,To help their unemployed father make ends mee...
5,The Unrecovered (2007),drama,The film's title refers not only to the un-re...


In [3]:
print(train_df.isnull().sum())
print(test_df.isnull().sum())
genres = train_df['genre'].unique()
print(sorted(list(genres)))
genre_counts = train_df['genre'].value_counts()
print(genre_counts)
genre_pct = train_df['genre'].value_counts(normalize=True) * 100
genre_dist_df = pd.DataFrame({'Count': genre_counts, 'Percentage (%)': genre_pct})
display(genre_dist_df)

title          0
genre          0
description    0
dtype: int64
title          0
description    0
dtype: int64
[' action ', ' adult ', ' adventure ', ' animation ', ' biography ', ' comedy ', ' crime ', ' documentary ', ' drama ', ' family ', ' fantasy ', ' game-show ', ' history ', ' horror ', ' music ', ' musical ', ' mystery ', ' news ', ' reality-tv ', ' romance ', ' sci-fi ', ' short ', ' sport ', ' talk-show ', ' thriller ', ' war ', ' western ']
genre
drama           13613
documentary     13096
comedy           7447
short            5073
horror           2204
thriller         1591
action           1315
western          1032
reality-tv        884
family            784
adventure         775
music             731
romance           672
sci-fi            647
adult             590
crime             505
animation         498
sport             432
talk-show         391
fantasy           323
mystery           319
musical           277
biography         265
history           243
game-show

,Count,Percentage (%)
genre,,
drama,13613,25.109750
documentary,13096,24.156122
comedy,7447,13.736304
short,5073,9.357362
horror,2204,4.065371
thriller,1591,2.934666
action,1315,2.425573
western,1032,1.903567
reality-tv,884,1.630575


In [4]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

t0 = time.time()
train_df['clean_description'] = train_df['description'].apply(clean_text)
test_df['clean_description'] = test_df['description'].apply(clean_text)
print(time.time() - t0)

6.333009958267212


In [5]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words='english')
X = tfidf.fit_transform(train_df['clean_description'])
y = train_df['genre']
X_test = tfidf.transform(test_df['clean_description'])
print(X.shape)
print(X_test.shape)

(54214, 20000)
(54200, 20000)


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_preds = nb_model.predict(X_val)
print(accuracy_score(y_val, nb_preds))

svm_model = LinearSVC(C=1.0, random_state=42, max_iter=2000)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_val)
print(accuracy_score(y_val, svm_preds))

0.5031817762611823
0.5703218666420732


In [7]:
print(classification_report(y_val, nb_preds, zero_division=0))
print(classification_report(y_val, svm_preds, zero_division=0))

               precision    recall  f1-score   support

      action        1.00      0.02      0.03       263
       adult        0.33      0.01      0.02       118
   adventure        0.60      0.02      0.04       155
   animation        0.00      0.00      0.00       100
   biography        0.00      0.00      0.00        53
      comedy        0.54      0.38      0.44      1490
       crime        0.00      0.00      0.00       101
 documentary        0.56      0.91      0.69      2619
       drama        0.44      0.85      0.58      2723
      family        0.00      0.00      0.00       157
     fantasy        0.00      0.00      0.00        65
   game-show        0.00      0.00      0.00        39
     history        0.00      0.00      0.00        49
      horror        0.82      0.15      0.26       441
       music        1.00      0.01      0.03       146
     musical        0.00      0.00      0.00        55
     mystery        0.00      0.00      0.00        64
        n

In [8]:
test_predictions = svm_model.predict(X_test)
test_df['predicted_genre'] = test_predictions
sample_preds = test_df[['title', 'predicted_genre', 'description']].sample(5, random_state=123)
display(sample_preds)
test_df[['title', 'predicted_genre']].to_csv('test_predictions.csv')

,title,predicted_genre,description
id,,,
27065,The Big Appeal (2017),short,"10 years old with a winning smile, Holly is C..."
47158,"CGI & Design of 'I, Robot' (2005)",documentary,This documentary delves headlong into the vis...
6366,A Tiger in the Dark: The Decadence Saga (2014),action,This is the story of a secret plot concocted ...
53750,Sad Digression (????),drama,"Michelle, a graduate from high school is awar..."
22005,Lincoln-Douglas Galesburg Debate (1994),drama,"Stephen A. Douglas, the incumbent senator, an..."


In [9]:
def predict_genre(description):
    cleaned = clean_text(description)
    vector = tfidf.transform([cleaned])
    prediction = svm_model.predict(vector)[0]
    print(prediction)

predict_genre("In a futuristic city, an astronaut sets off on a dangerous space flight to an uncharted planet to find alternative energy sources to save a dying Earth.")
predict_genre("A group of young teenagers decide to stay the night in a cursed cabin in the deep woods, only to realize a dark spirit haunts the halls and hunts them.")
predict_genre("Two childhood sweetheart lovers from rival families reunite in a small Italian town, attempting to overcome familial hatred to start a life together.")

 sci-fi 
 horror 
 drama 
